In [7]:
import torch
import torch.nn as nn
import numpy as np
import torch.nn.functional as F
import kagglehub

# Download latest version
path = kagglehub.dataset_download("bhavikjikadara/dog-and-cat-classification-dataset")

print("Path to dataset files:", path)

Using Colab cache for faster access to the 'dog-and-cat-classification-dataset' dataset.
Path to dataset files: /kaggle/input/dog-and-cat-classification-dataset


In [8]:
!git clone https://github.com/MohamedBenloughmari/VIT-Implementation.git

Cloning into 'VIT-Implementation'...
remote: Enumerating objects: 39, done.
remote: Counting objects: 100% (39/39), done.
remote: Compressing objects: 100% (28/28), done.
remote: Total 39 (delta 15), reused 31 (delta 10), pack-reused 0 (from 0)
Receiving objects: 100% (39/39), 23.86 KiB | 23.86 MiB/s, done.
Resolving deltas: 100% (15/15), done.


In [9]:
%cd VIT-Implementation

/content/VIT-Implementation/VIT-Implementation


In [10]:
import os
path=os.path.join(path,'PetImages')

In [11]:
import os
import torchvision.transforms as transforms
from torch.utils.data import Dataset, DataLoader
from PIL import Image
from Preprocessing import AutoTokenizer
tokenizer=AutoTokenizer()
class CustomImageDataset(Dataset):
    def __init__(self, root_dir, transform=None):
        self.root_dir = root_dir
        self.transform = transform
        self.classes = sorted([d for d in os.listdir(root_dir) if os.path.isdir(os.path.join(root_dir, d))])
        self.class_to_idx = {cls_name: i for i, cls_name in enumerate(self.classes)}
        self.samples = []
        
        for class_name in self.classes:
            class_path = os.path.join(root_dir, class_name)
            for img_name in os.listdir(class_path):
                self.samples.append((os.path.join(class_path, img_name), self.class_to_idx[class_name]))
    
    def __len__(self):
        return len(self.samples)
    
    def __getitem__(self, idx):
        img_path, label = self.samples[idx]
        image = Image.open(img_path).convert('RGB')
        
        if self.transform:
            image = self.transform(image)
        return image, label

# Define transforms
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                        std=[0.229, 0.224, 0.225])
])

# Load dataset
dataset = CustomImageDataset(root_dir=path, transform=transform)

# Split dataset into train and validation sets
from sklearn.model_selection import train_test_split

train_samples, val_samples = train_test_split(dataset.samples, 
                                            test_size=0.2, 
                                            random_state=42,
                                            stratify=[s[1] for s in dataset.samples])

train_dataset = CustomImageDataset(root_dir=path, 
                                 transform=transform)
val_dataset = CustomImageDataset(root_dir=path, 
                               transform=transform)

train_dataset.samples = train_samples
val_dataset.samples = val_samples

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False)
#device=torch.device('mps' if torch.backends.mps.is_available() else 'cpu')
device=torch.device('cuda')

In [12]:
from models import Transformer
from torch.optim import Adam
from torch.optim.lr_scheduler import CosineAnnealingWarmRestarts, LinearLR, SequentialLR
from tqdm import tqdm 
class binary_classifier(nn.Module):
    def __init__(self,input_dim=None,n_labels=None):
        super().__init__()
        self.n_labels=n_labels
        self.input_dim=input_dim
        self.transformer=Transformer(input_dim=input_dim)
        self.classifier=nn.Sequential(
            nn.Linear(588,32),
            nn.Dropout(0.3),
            nn.ReLU(),
            nn.Linear(32,16),
            nn.ReLU(),
            nn.Linear(16,n_labels)
        )
    def forward(self,x):
        x=self.transformer(x)
        x = x.mean(dim=1) 
        return self.classifier(x)
model=binary_classifier(input_dim=588,n_labels=2)
model.to(device)
tokenizer.to(device)
optimizer=Adam(params=list(model.parameters())+list(tokenizer.parameters()),lr=3e-4,weight_decay=0.01)
num_epochs=10
total_steps = num_epochs * len(train_loader)
warmup_steps = int(0.1 * total_steps)
warmup_scheduler = LinearLR(optimizer, start_factor=0.01, end_factor=1.0, total_iters=warmup_steps)
cosine_scheduler = CosineAnnealingWarmRestarts(optimizer, T_0=total_steps - warmup_steps)
scheduler = SequentialLR(optimizer, schedulers=[warmup_scheduler, cosine_scheduler], milestones=[warmup_steps])
criterion=torch.nn.CrossEntropyLoss()
for epoch in range(num_epochs):
    total_true=0
    tota_seq=0
    y_true=[]
    y_pred=[]
    pbar= tqdm(train_loader,desc=f"Epoch{epoch}/{num_epochs}",leave=True)
    for image , label in pbar:
        image=image.to(device)
        label=label.to(device)
        token=tokenizer.tokenize(img=image)
        optimizer.zero_grad()
        output=model(token)
        _,predicted=torch.max(output,1)
        predicted=predicted.long()
        loss=criterion(output,label)
        total_true+=(predicted==label).sum()
        tota_seq+=token.shape[0]
        loss.backward()
        optimizer.step()
        scheduler.step()
    print(total_true/tota_seq)

Epoch0/10: 100%|██████████| 625/625 [06:05<00:00,  1.71it/s]


tensor(0.5247, device='cuda:0')


Epoch1/10: 100%|██████████| 625/625 [05:59<00:00,  1.74it/s]


tensor(0.5463, device='cuda:0')


Epoch2/10: 100%|██████████| 625/625 [05:55<00:00,  1.76it/s]


tensor(0.5546, device='cuda:0')


Epoch3/10: 100%|██████████| 625/625 [05:41<00:00,  1.83it/s]


tensor(0.5530, device='cuda:0')


Epoch4/10:   2%|▏         | 11/625 [00:05<05:03,  2.02it/s]


KeyboardInterrupt: 